# ParcelWatch v1: fine-tune the 4-band Mask R-CNN on Colab

Runtime > Change runtime type > **T4 GPU**. Then run every cell top to bottom (about 2 to 3 hours).

Before you start, upload two files to the Google Drive folder `MyDrive/parcelwatch/`:

1. `dist/parcelwatch_src.zip`
2. `dist/v1_tiles.zip`

Both are made on the laptop by `python scripts/pack_for_colab.py`.

When it finishes, download `MyDrive/parcelwatch/models/v1/best_model.pth` and
`train_settings.json` into `D:/ParcelWatch/models/v1/`. All settings come from
`configs/wake_pilot.yaml` (50 epochs, batch 4, learning rate 0.005, seed 42).

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/parcelwatch"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!mkdir -p /content/pw/data/training && cd /content/pw && unzip -q -o $DRIVE/parcelwatch_src.zip && unzip -q -o $DRIVE/v1_tiles.zip -d data/training/
!cd /content/pw && pip -q install -e ".[ml]"

In [ ]:
import pathlib
import sys

sys.path.insert(0, "/content/pw/src")
from parcelwatch.config import load_config
from parcelwatch.detect import train_v1

cfg = load_config("/content/pw/configs/wake_pilot.yaml")
tiles = pathlib.Path("/content/pw/data/training/v1_tiles")
print(len(list((tiles / "images").glob("*.tif"))), "training tiles")
print(cfg.model.v1)

In [ ]:
# Checkpoints go straight to Drive, so a disconnect loses nothing.
out = pathlib.Path(DRIVE) / "models" / "v1"
best = train_v1(tiles, out, cfg)
print("best weights:", best)

Done. Download `MyDrive/parcelwatch/models/v1/best_model.pth` and `train_settings.json` to
`D:/ParcelWatch/models/v1/`, then tell Claude: "Colab training finished, best_model.pth is in models/v1."